In [1]:
# =========================
# nb_ingest_pershing
# =========================

# ---------- Parameters ----------
period = "2026-03"            # YYYY-MM
run_id = "manual_test_run"    # passed by nb_run_all in real run

dfm_id = "pershing"
dfm_name = "Pershing"

# ---------- Imports ----------
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType
import re

# ---------- Paths ----------
landing_path = f"Files/landing/period={period}/dfm={dfm_id}/source/"
fx_path = "Files/config/fx_rates.csv"

# ---------- Helpers ----------
def q(s: str) -> str:
    return (s or "").replace("'", "''")

def norm(c: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")

def dec_col(c, p=28, s=10):
    return F.col(c).cast(f"decimal({p},{s})")

def write_audit(files_processed, rows_ingested, parse_errors_count, drift_events_count, status):
    spark.sql(f"""
        INSERT INTO run_audit_log
        (run_id, period, dfm_id, files_processed, rows_ingested, parse_errors_count, drift_events_count, status, started_at, completed_at)
        VALUES
        ('{q(run_id)}','{q(period)}','{q(dfm_id)}',{int(files_processed)},{int(rows_ingested)},{int(parse_errors_count)},{int(drift_events_count)},'{q(status)}',current_timestamp(),current_timestamp())
    """)

def write_parse_errors(df_err):
    cols = set(df_err.columns)
    out = df_err

    if "source_file" not in cols:
        out = out.withColumn("source_file", F.lit("UNKNOWN"))
    if "row_number" not in cols:
        if "source_row_id" in cols:
            out = out.withColumn("row_number", F.regexp_extract(F.col("source_row_id").cast("string"), r"(\\d+)$", 1).cast("bigint"))
        else:
            out = out.withColumn("row_number", F.lit(None).cast("bigint"))
    if "column_name" not in cols:
        out = out.withColumn("column_name", F.lit(None).cast("string"))
    if "raw_value" not in cols:
        out = out.withColumn("raw_value", F.lit(None).cast("string"))
    if "error_type" not in cols:
        out = out.withColumn("error_type", F.lit("PARSE_ERROR"))
    if "error_message" not in cols:
        out = out.withColumn("error_message", F.lit("Unknown parse error"))

    (
        out.withColumn("period", F.lit(period))
           .withColumn("run_id", F.lit(run_id))
           .withColumn("dfm_id", F.lit(dfm_id))
           .withColumn("event_ts", F.current_timestamp())
           .select("period","run_id","dfm_id","source_file","row_number","column_name","raw_value","error_type","error_message","event_ts")
           .write.mode("append").saveAsTable("parse_errors")
    )

def read_csv_with_source(path, source_name):
    df = spark.read.option("header", True).csv(path)
    for c in df.columns:
        df = df.withColumnRenamed(c, norm(c))
    return df.withColumn("source_file", F.lit(source_name))

# ---------- A3: File discovery ----------
try:
    entries = [f for f in mssparkutils.fs.ls(landing_path) if not f.isDir]
except Exception:
    write_audit(0, 0, 0, 0, "NO_FILES")
    mssparkutils.notebook.exit("NO_FILES")

if len(entries) == 0:
    write_audit(0, 0, 0, 0, "NO_FILES")
    mssparkutils.notebook.exit("NO_FILES")

positions_files = [f for f in entries if f.name.lower() == "positions.csv"]
valuation_files = [f for f in entries if f.name.lower().startswith("psl_valuation_holdings_") and f.name.lower().endswith(".csv")]

if len(positions_files) == 0 and len(valuation_files) == 0:
    write_audit(len(entries), 0, 0, 0, "NO_FILES")
    mssparkutils.notebook.exit("NO_FILES")

# ---------- A4: Parse source files ----------
df_pos = None
if positions_files:
    pfile = positions_files[0]
    df_pos = read_csv_with_source(pfile.path, pfile.name)
    w = Window.orderBy(F.monotonically_increasing_id())
    df_pos = (
        df_pos
        .withColumn("_rn", F.row_number().over(w))
        .withColumn("source_row_id_pos", F.concat_ws(":", F.col("source_file"), F.col("_rn").cast("string")))
        .drop("_rn")
    )

df_val = None
if valuation_files:
    val_parts = []
    for vfile in valuation_files:
        one = read_csv_with_source(vfile.path, vfile.name)
        w = Window.orderBy(F.monotonically_increasing_id())
        one = (
            one
            .withColumn("_rn", F.row_number().over(w))
            .withColumn("source_row_id_val", F.concat_ws(":", F.col("source_file"), F.col("_rn").cast("string")))
            .drop("_rn")
        )
        val_parts.append(one)

    df_val = val_parts[0]
    for nxt in val_parts[1:]:
        df_val = df_val.unionByName(nxt, allowMissingColumns=True)

# ---------- Column contract checks ----------
parse_err_seed = []

pos_expected = [
    "accountnumber","portfolionumber","reportdate","positiondate","isin",
    "quantity","price","localcurrencyiso","marketvalue",
    "reportingmarketvalue","reportingmarketvalueiso","fxrate"
]

val_expected = [
    "pslaccountreference","valuedate","holding","price","value","ccyoftheasset","accruedinterest"
]

if df_pos is not None:
    missing = [c for c in pos_expected if c not in df_pos.columns]
    if missing:
        parse_err_seed.append((positions_files[0].name, None, "header", "", "MISSING_COLUMN", f"Positions.csv missing: {','.join(missing)}"))

if df_val is not None:
    missing = [c for c in val_expected if c not in df_val.columns]
    if missing:
        parse_err_seed.append(("PSL_Valuation_Holdings_*", None, "header", "", "MISSING_COLUMN", f"Valuation file missing: {','.join(missing)}"))

if parse_err_seed:
    seed_schema = StructType([
        StructField("source_file", StringType(), True),
        StructField("row_number", LongType(), True),
        StructField("column_name", StringType(), True),
        StructField("raw_value", StringType(), True),
        StructField("error_type", StringType(), True),
        StructField("error_message", StringType(), True),
    ])
    seed_rows = [
        (
            str(r[0]) if r[0] is not None else None,
            int(r[1]) if r[1] is not None else None,
            str(r[2]) if r[2] is not None else None,
            str(r[3]) if r[3] is not None else None,
            str(r[4]) if r[4] is not None else "PARSE_ERROR",
            str(r[5]) if r[5] is not None else "Unknown parse error",
        )
        for r in parse_err_seed
    ]
    write_parse_errors(spark.createDataFrame(seed_rows, seed_schema))

# ---------- Primary mapping (Positions.csv) ----------
if df_pos is not None:
    p = (
        df_pos
        .withColumn("policy_id_pos", F.coalesce(F.col("accountnumber"), F.col("portfolionumber")).cast("string"))
        .withColumn("report_date_pos", F.to_date(F.coalesce(F.col("reportdate"), F.col("positiondate"))))
        .withColumn("isin_pos", F.col("isin").cast("string"))
        .withColumn("holding_pos", F.col("quantity").cast("double"))
        .withColumn("local_bid_price_pos", F.col("price").cast("double"))
        .withColumn("local_currency_pos", F.upper(F.trim(F.col("localcurrencyiso").cast("string"))))
        .withColumn("bid_value_local_pos", F.col("marketvalue").cast("double"))
        .withColumn("reporting_mv_pos", F.col("reportingmarketvalue").cast("double"))
        .withColumn("reporting_iso_pos", F.upper(F.trim(F.col("reportingmarketvalueiso").cast("string"))))
        .withColumn("fxrate_pos", F.col("fxrate").cast("double"))
        .withColumn("accrued_interest_local_pos", F.lit(None).cast("double"))
    )
else:
    p = spark.createDataFrame([], """
        source_file string, source_row_id_pos string,
        policy_id_pos string, report_date_pos date, isin_pos string,
        holding_pos double, local_bid_price_pos double, local_currency_pos string,
        bid_value_local_pos double, reporting_mv_pos double, reporting_iso_pos string,
        fxrate_pos double, accrued_interest_local_pos double
    """)

# ---------- Secondary mapping (valuation) ----------
if df_val is not None:
    v = (
        df_val
        .withColumn("policy_id_val", F.col("pslaccountreference").cast("string"))
        .withColumn("report_date_val", F.to_date(F.col("valuedate")))
        .withColumn("holding_val", F.col("holding").cast("double"))
        .withColumn("local_bid_price_val", F.col("price").cast("double"))
        .withColumn("bid_value_local_val", F.col("value").cast("double"))
        .withColumn("local_currency_val", F.upper(F.trim(F.col("ccyoftheasset").cast("string"))))
        .withColumn("accrued_interest_local_val", F.col("accruedinterest").cast("double"))
    )
else:
    v = spark.createDataFrame([], """
        source_file string, source_row_id_val string,
        policy_id_val string, report_date_val date,
        holding_val double, local_bid_price_val double, bid_value_local_val double,
        local_currency_val string, accrued_interest_local_val double
    """)

# ---------- Join + backfill ----------
joined = (
    p.alias("p")
    .join(
        v.alias("v"),
        on=[
            F.col("p.policy_id_pos") == F.col("v.policy_id_val"),
            F.col("p.report_date_pos") == F.col("v.report_date_val")
        ],
        how="full_outer"
    )
)

base = (
    joined
    .withColumn("policy_id", F.coalesce(F.col("p.policy_id_pos"), F.col("v.policy_id_val")))
    .withColumn("report_date", F.coalesce(F.col("p.report_date_pos"), F.col("v.report_date_val")))

    # disambiguated canonical columns
    .withColumn("isin_out", F.col("p.isin_pos").cast("string"))
    .withColumn("security_id_out", F.col("p.isin_pos").cast("string"))
    .withColumn("holding_out", F.coalesce(F.col("p.holding_pos"), F.col("v.holding_val")))
    .withColumn("local_bid_price_out", F.coalesce(F.col("p.local_bid_price_pos"), F.col("v.local_bid_price_val")))
    .withColumn("bid_value_local_out", F.coalesce(F.col("p.bid_value_local_pos"), F.col("v.bid_value_local_val")))
    .withColumn("local_currency_out", F.coalesce(F.col("p.local_currency_pos"), F.col("v.local_currency_val")))
    .withColumn("accrued_interest_local_out", F.coalesce(F.col("p.accrued_interest_local_pos"), F.col("v.accrued_interest_local_val")))

    .withColumn("source_file_out", F.coalesce(F.col("p.source_file"), F.col("v.source_file")))
    .withColumn("source_row_id_raw", F.coalesce(F.col("p.source_row_id_pos"), F.col("v.source_row_id_val")))

    .withColumn("reporting_mv", F.col("p.reporting_mv_pos"))
    .withColumn("reporting_iso", F.col("p.reporting_iso_pos"))
    .withColumn("fxrate_pos", F.col("p.fxrate_pos"))
    .withColumn("f_backfilled", F.when(F.col("p.policy_id_pos").isNull() & F.col("v.policy_id_val").isNotNull(), F.lit("BACKFILLED_FROM_VALUATION")))
)

# ---------- FX ----------
fx = (
    spark.read.option("header", True).csv(fx_path)
    .withColumn("currency_u", F.upper(F.trim(F.col("currency"))))
    .withColumn("rate_to_gbp_d", F.col("rate_to_gbp").cast("double"))
    .select("currency_u", "rate_to_gbp_d")
)

base = base.join(fx, F.col("local_currency_out") == F.col("currency_u"), "left")

# ---------- GBP logic ----------
base = (
    base
    .withColumn(
        "bid_value_gbp_d",
        F.when(F.col("reporting_iso") == "GBP", F.col("reporting_mv"))
         .when(F.col("local_currency_out") == "GBP", F.col("bid_value_local_out"))
         .when((F.col("fxrate_pos").isNotNull()) & (F.col("reporting_iso") == "GBP"), F.col("bid_value_local_out") * F.col("fxrate_pos"))
         .when((F.col("rate_to_gbp_d").isNotNull()) & F.col("bid_value_local_out").isNotNull(), F.col("bid_value_local_out") * F.col("rate_to_gbp_d"))
         .otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "accrued_interest_gbp_d",
        F.when(F.col("local_currency_out") == "GBP", F.col("accrued_interest_local_out"))
         .when((F.col("rate_to_gbp_d").isNotNull()) & F.col("accrued_interest_local_out").isNotNull(), F.col("accrued_interest_local_out") * F.col("rate_to_gbp_d"))
         .otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "fx_rate_out",
        F.when(F.col("local_currency_out") == "GBP", F.lit(1.0))
         .when(F.col("fxrate_pos").isNotNull(), F.col("fxrate_pos"))
         .otherwise(F.col("rate_to_gbp_d"))
    )
    .withColumn("cash_value_gbp_d", F.lit(0.0))
    .withColumn("f_fx_assumed", F.when((F.col("fxrate_pos").isNotNull()) & (F.col("reporting_iso") == "GBP") & (F.col("local_currency_out") != "GBP"), F.lit("FX_RATE_ASSUMED")))
    .withColumn("f_fx_na", F.when(F.col("bid_value_gbp_d").isNull(), F.lit("FX_NOT_AVAILABLE")))
    .withColumn("f_accrued_na", F.when(F.col("accrued_interest_gbp_d").isNull(), F.lit("ACCRUED_NOT_AVAILABLE")))
    .withColumn("data_quality_flags", F.expr("filter(array(f_backfilled, f_fx_assumed, f_fx_na, f_accrued_na), x -> x is not null)"))
)

# ---------- Quality errors ----------
bad = (
    base
    .filter(F.col("policy_id").isNull() | F.col("report_date").isNull())
    .select(
        F.col("source_file_out").alias("source_file"),
        F.coalesce(F.col("source_row_id_raw"), F.lit("UNKNOWN")).alias("source_row_id"),
        F.lit("Required field missing: policy_id/report_date").alias("error_message"),
        F.concat_ws("|", F.col("policy_id"), F.col("report_date").cast("string")).alias("raw_value")
    )
)

bad_count = bad.count()
if bad_count > 0:
    write_parse_errors(
        bad
        .withColumn("column_name", F.lit("policy_id/report_date"))
        .withColumn("error_type", F.lit("MISSING_REQUIRED"))
    )
good = base.filter(F.col("policy_id").isNotNull() & F.col("report_date").isNotNull())

# ---------- Stable row hash ----------
good = good.withColumn(
    "source_row_id",
    F.sha2(
        F.concat_ws(
            "|",
            F.lit(period), F.lit(dfm_id),
            F.col("policy_id"),
            F.coalesce(F.col("isin_out"), F.lit("")),
            F.col("report_date").cast("string"),
            F.coalesce(F.col("holding_out").cast("string"), F.lit("")),
            F.coalesce(F.col("bid_value_local_out").cast("string"), F.lit("")),
            F.coalesce(F.col("source_file_out"), F.lit("")),
            F.coalesce(F.col("source_row_id_raw"), F.lit(""))
        ),
        256
    )
)
good_clean = good.select(
    F.col("policy_id"),
    F.col("report_date"),
    F.col("source_row_id"),
    F.col("source_file_out").alias("source_file"),
    F.col("security_id_out").alias("security_id"),
    F.col("isin_out").alias("isin"),
    F.col("holding_out").alias("holding"),
    F.col("local_bid_price_out").alias("local_bid_price"),
    F.col("local_currency_out").alias("local_currency"),
    F.col("fx_rate_out").alias("fx_rate"),
    F.col("cash_value_gbp_d").alias("cash_value_gbp"),
    F.col("bid_value_gbp_d").alias("bid_value_gbp"),
    F.col("accrued_interest_gbp_d").alias("accrued_interest_gbp"),
    F.col("data_quality_flags")
)

# ---------- Canonical projection ----------
canonical = (
    good_clean
    .withColumn("period", F.lit(period))
    .withColumn("run_id", F.lit(run_id))
    .withColumn("dfm_id", F.lit(dfm_id))
    .withColumn("dfm_name", F.lit(dfm_name))
    .withColumn("source_sheet", F.lit(None).cast("string"))
    .withColumn("policy_id_type", F.lit("DFM"))
    .withColumn("dfm_policy_id", F.col("policy_id"))
    .withColumn("other_security_id", F.lit(None).cast("string"))
    .withColumn("id_type", F.when(F.col("isin").isNotNull(), F.lit("ISIN")).otherwise(F.lit(None).cast("string")))
    .withColumn("asset_name", F.lit(None).cast("string"))
    .withColumn("ingested_at", F.current_timestamp())
    .select(
        "period","run_id","dfm_id","dfm_name",
        "source_file","source_sheet","source_row_id",
        "policy_id","policy_id_type","dfm_policy_id",
        "security_id","isin","other_security_id","id_type","asset_name",
        dec_col("holding").alias("holding"),
        dec_col("local_bid_price").alias("local_bid_price"),
        "local_currency",
        dec_col("fx_rate").alias("fx_rate"),
        dec_col("cash_value_gbp").alias("cash_value_gbp"),
        dec_col("bid_value_gbp").alias("bid_value_gbp"),
        dec_col("accrued_interest_gbp").alias("accrued_interest_gbp"),
        "report_date","ingested_at","data_quality_flags"
    )
    .dropDuplicates(["run_id", "source_row_id"])
)


# Align dataframe column types to the existing canonical_holdings table schema
target_schema = spark.table("canonical_holdings").schema
target_cols = [f.name for f in target_schema]

# cast each existing column to target type
for f in target_schema:
    if f.name in canonical.columns:
        canonical = canonical.withColumn(f.name, F.col(f.name).cast(f.dataType))
    else:
        canonical = canonical.withColumn(f.name, F.lit(None).cast(f.dataType))

# reorder exactly to table order
canonical = canonical.select(*target_cols)

# ---------- Write ----------
canonical.write.mode("append").saveAsTable("canonical_holdings")
rows_ingested = canonical.count()

# ---------- Audit ----------
status = "PARTIAL" if (bad_count + len(parse_err_seed)) > 0 else "OK"
write_audit(
    files_processed=len(entries),
    rows_ingested=rows_ingested,
    parse_errors_count=bad_count + len(parse_err_seed),
    drift_events_count=0,
    status=status
)

mssparkutils.notebook.exit(status)

StatementMeta(, 2b8bb8b3-3d77-401f-bbc8-ff2ba49d4936, 3, Finished, Available, Finished, False)

ExitValue: PARTIAL